# Getting Started with OpenAgent

This notebook demonstrates how to create and use an OpenAgent agent.

OpenAgent is a general-purpose agent harness that gives LLM agents access to a computer via the terminal to complete tasks the way humans do.

We'll use **E2B cloud sandboxes** for secure, isolated execution.

## Helper Function for Streaming

We'll use streaming throughout this notebook to show responses in real-time.

Color-coded headers:
- **Yellow**: `<Human>` - User messages
- **Magenta**: `<AI (Thinking)>` - Assistant reasoning
- **Blue**: `<AI (Text)>` - Assistant responses
- **Green**: `<ToolCall: name>` - Tool invocations
- **Cyan**: `<ToolResult: name>` - Tool output
- **Red**: Error messages

In [13]:
import json

from langchain_core.messages import AIMessageChunk, HumanMessage, ToolMessage
from langgraph.graph.state import CompiledStateGraph

# ANSI color codes
RESET = "\033[0m"
RED = "\033[31m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
MAGENTA = "\033[35m"
CYAN = "\033[36m"


def format_json(obj: dict | list | str) -> str:
    """Format object as readable JSON."""
    if isinstance(obj, str):
        return obj
    return json.dumps(obj, indent=2, ensure_ascii=False)


async def stream_agent(agent: CompiledStateGraph, message: str, config: dict | None = None) -> list:
    """Stream agent response with real-time output.

    Uses astream() with stream_mode="messages" for direct message streaming.
    Handles content_blocks properly:
    - "reasoning": Extended thinking/reasoning content
    - "text": Regular text content
    - "tool_use": Tool invocations
    - "tool_call_chunk": Streaming tool call chunks

    Returns:
        List of all (chunk, metadata) tuples streamed from the agent.
    """
    print(f"{YELLOW}<Human>{RESET}")
    print(message)
    print()

    inputs = {"messages": [HumanMessage(content=message)]}

    current_block = None  # "text", "reasoning", "tool_call", or None
    results = []

    async for chunk, metadata in agent.astream(inputs, config=config, stream_mode="messages"):
        results.append((chunk, metadata))

        # Handle ToolMessage (tool results)
        if isinstance(chunk, ToolMessage):
            if current_block:
                print("\n")
                current_block = None
            print(f"{CYAN}<ToolResult: {chunk.name}>{RESET}")
            print(str(chunk.content) if chunk.content else "(empty)")
            print()
            continue

        # Only process AI message chunks
        if not isinstance(chunk, AIMessageChunk):
            continue

        # Use content_blocks for proper streaming (preferred)
        content_blocks = getattr(chunk, "content_blocks", None)
        if content_blocks:
            for block in content_blocks:
                if not isinstance(block, dict):
                    continue

                block_type = block.get("type")

                if block_type == "reasoning":
                    # Extended thinking/reasoning content
                    reasoning = block.get("reasoning")
                    if reasoning:
                        if current_block != "reasoning":
                            if current_block:
                                print("\n")
                            print(f"{MAGENTA}<AI (Thinking)>{RESET}")
                            current_block = "reasoning"
                        print(reasoning, end="", flush=True)

                elif block_type == "text":
                    # Regular text content
                    text = block.get("text")
                    if text:
                        if current_block != "text":
                            if current_block:
                                print("\n")
                            print(f"{BLUE}<AI (Text)>{RESET}")
                            current_block = "text"
                        print(text, end="", flush=True)

                elif block_type == "tool_use":
                    # Complete tool invocation
                    if current_block:
                        print("\n")
                    print(f"{GREEN}<ToolCall: {block.get('name', 'unknown')}>{RESET}")
                    print(format_json(block.get("input", {})))
                    print()
                    current_block = None

                elif block_type == "tool_call_chunk":
                    # Streaming tool call chunk
                    name = block.get("name")
                    args = block.get("args")
                    if name:
                        # New tool call starting
                        if current_block:
                            print("\n")
                        print(f"{GREEN}<ToolCall: {name}>{RESET}")
                        current_block = "tool_call"
                    if args:
                        print(args, end="", flush=True)

            continue

        # Fallback: handle content directly (for models without content_blocks)
        content = chunk.content
        if isinstance(content, str) and content:
            if current_block != "text":
                if current_block:
                    print("\n")
                print(f"{BLUE}<AI (Text)>{RESET}")
                current_block = "text"
            print(content, end="", flush=True)

    # End any open block
    if current_block:
        print("\n")

    return results

## Creating a Cloud Sandbox

We use `RemoteE2BComputer` to run commands in a secure cloud sandbox. The sandbox:
- Provides isolated Linux environment
- Persists state (files, packages) across commands
- Auto-pauses when idle to save costs
- Can be resumed later with the same `sandbox_id`

In [14]:
from openagent.computer import RemoteE2BComputer
from openagent.langchain import create_agent

# Create a cloud sandbox with 3 minute lifetime
# Template "openagent" provides 2 CPU, 4 GiB memory ($0.17/hr)
computer = RemoteE2BComputer(template="openagent", lifetime=180)
await computer.start()

print(f"Sandbox ID: {computer.sandbox_id}")
print("(Save this ID to reconnect later)")

Sandbox ID: ih2tw2i3k7qwy61fpp4pg
(Save this ID to reconnect later)


In [ ]:
import os
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="kimi-k2.5",
    api_key=os.environ["UNICOM_API_KEY"],
    api_base=os.environ["UNICOM_BASE_URL"],
)

In [16]:
# Create the agent with default settings
agent = create_agent(computer=computer, model=model)

In [7]:
# Stream agent response in real-time
original_events = await stream_agent(
    agent, "Check what OS it is. Then write a summary of the OS to file. After that, try the best you can to convert the file to a PDF."
)

<Human>
Check what OS it is. Then write a summary of the OS to file. After that, try the best you can to convert the file to a PDF.

<AI (Thinking)>
 The user wants me to:
1. Check what OS it is
2. Write a summary of the OS to a file
3. Try to convert the file to a PDF

Let me start by checking the OS using bash commands.

<ToolCall: bash>
{"command": "uname -a && cat /etc/os-release 2>/dev/null || cat /etc/*release 2>/dev/null"}

<ToolResult: bash>
Linux e2b.local 6.1.158 #2 SMP PREEMPT_DYNAMIC Tue Nov 25 15:58:27 UTC 2025 x86_64 x86_64 x86_64 GNU/Linux
PRETTY_NAME="Ubuntu 24.04.3 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.3 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo

<AI (Thinking)>
 The OS is Ubuntu 24.0

## Running Multi-Turn Conversations

The agent maintains state across multiple invocations when using a checkpointer.

In [21]:
from langgraph.checkpoint.memory import MemorySaver

# Create agent with checkpointer for persistent memory
checkpointer = MemorySaver()
agent = create_agent(computer, model, checkpointer=checkpointer)

# Session config for multi-turn
config = {"configurable": {"thread_id": "my-session"}}

In [22]:
# First turn - install a package (persists in sandbox)
_ = await stream_agent(agent, "Install the 'cowsay' package with pip (using --break-system-packages)", config=config)

<Human>
Install the 'cowsay' package with pip (using --break-system-packages)

<AI (Thinking)>
 The user wants me to install the 'cowsay' package using pip with the --break-system-packages flag. I'll use the bash tool to run the pip install command.

<ToolCall: bash>
{"command": "pip install cowsay --break-system-packages"}

<AI (Text)>
  

<ToolResult: bash>

<AI (Thinking)>
 The `cowsay` package is already installed. The task is complete.

<AI (Text)>
 The `cowsay` package is already installed on the system at `/usr/local/lib/python3.12/dist-packages` (version 6.1). The installation command ran successfully and confirmed that the requirement was already satisfied.



In [23]:
# Second turn - use the installed package
_ = await stream_agent(agent, "Use cowsay to say 'Hello from E2B!'", config=config)

<Human>
Use cowsay to say 'Hello from E2B!'

<AI (Thinking)>
 The user wants me to use cowsay to display a message saying "Hello from E2B!". Since cowsay is a Python package installed via pip, I need to use Python to run it. I'll use the command `python -c "import cowsay; cowsay.cow('Hello from E2B!')"` or just run `cowsay` directly if it's available as a CLI.

<ToolCall: bash>
{"command": "python -c \"import cowsay; cowsay.cow('Hello from E2B!')\""}

<AI (Text)>
  

<ToolResult: bash>
<error>Exit code 127
/bin/bash: line 1: python: command not found</error>

<AI (Thinking)>
 Let me try using `python3` instead:

<ToolCall: bash>
{"command": "python3 -c \"import cowsay; cowsay.cow('Hello from E2B!')\""}

<AI (Text)>
  

<ToolResult: bash>
  _______________
| Hello from E2B! |
               \
                \
                  ^__^
                  (oo)\_______
                  (__)\       )\/\
                      ||----w |
                      ||     ||

<AI (Thinking)>
 The cows

In [24]:
# Third turn - agent remembers what we did
_ = await stream_agent(agent, "What packages did we install in this session?", config=config)

<Human>
What packages did we install in this session?

<AI (Thinking)>
 The user is asking what packages were installed in this session. Based on the conversation:

1. The user asked to install `cowsay` package with pip using `--break-system-packages`
2. The pip install command showed "Requirement already satisfied: cowsay in /usr/local/lib/python3.12/dist-packages (6.1)"

So technically, we didn't install any new packages in this session - cowsay was already installed. However, we did run the pip install command which confirmed the package was available.

Let me provide an accurate answer based on the session history.

<AI (Text)>
 Based on this session, here's what happened with package installation:

**We did not install any new packages.** 

When we ran:
```bash
pip install cowsay --break-system-packages
```

The output was:
- **Requirement already satisfied**: cowsay in /usr/local/lib/python3.12/dist-packages (6.1)

This means `cowsay` was already installed on the system before th

## Cleanup

Stop the sandbox when done. This pauses it (preserving state) rather than destroying it.

In [25]:
await computer.stop()
print("Sandbox paused. State preserved for reconnection.")

Sandbox paused. State preserved for reconnection.
